In [ ]:
#Task 3.1
from flask import Flask, render_template, redirect, request, url_for
import sqlite3

app = Flask("__name__")
ac = ["POST", "GET"] #accepted methods

def getdb():
    with sqlite3.connect("WORKSHOPS.db") as conn:
        global conn
#         conn.row_factory = sqlite3.Row
        return conn.cursor()



@app.route("/", methods=ac)
def index():
    if method == "POST":
        return redirect({{url_for("INDEX.html")}})
    else: 
        workshops = getdb().execute("""
        SELECT *
        FROM Workshop
        ORDER BY Title ASC
        """).fetchall()
        conn.close()
        return render_template("INDEX.html", workshops = workshops)
    

In [ ]:
#Task 3.2
import datetime

@app.route("/REGISTER", methods=ac)
def register():
    if method == "POST":
        student_id = request.get("student_id", "")
        workshop_id = request.get("workshop_id", "")
        return redirect({{url_for("REGISTER.html")}})
    else:
        #filtering ids and returning updated list
        students = getdb().execute("""
        SELECT StudentID, StudentName
        FROM Student
        WHERE StudentID = ?
        ORDER BY StudentID ASC
        """, (student_id,)).fetchall()
        workshops = getdb().execute("""
        SELECT WorkshopID, Title
        FROM Workshop
        WHERE WorkshopID = ?
        ORDER BY WorkshopID ASC
        """, (workshop_id,)).fetchall()
        #find max places for workshop and current number of registrations 
        workshop_registry = get.db().execute("""
        SELECT Workshop.MaxPlaces, COUNT(*) AS current
        FROM Workshop INNER JOIN Registration ON Workshop.WorkshopID = Registration.WorkshopID
        GROUP BY Workshop.WorkshopID
        HAVING Workshop.WorkshopID = ?
        """, (workshop_id,)).fetchone()
        #find student's registrations as a list of workshop_id 
        student_registrations = get.db().execute("""
        SELECT Registration.WorkshopID
        FROM Student INNER JOIN Registration ON Student.StudentID = Registration.StudentID
        WHERE Student.StudentID = ?
        """, (student_id,)).fetchall()
        #perform comparison logic to handle edge cases
        #duplicate registration 
        if workshop_id in student_registrations:
            message = "this student has already registered for the workshop"
        #workshop is full 
        elif workshop_registry[1] >= workshop_registry[0]: #registrations greater equal to max capacity
            message = "this workshop is full"
        else: #edge cases passed; can register
            getdb().execute("""
            INSERT OR IGNORE INTO Registrations(WorkshopID, StudentID, RegistrationDate)
            VALUES (?,?,?)
            """,(workshop_id, student_id, datetime.now(),))
            conn.commit()
            message = "successful registration"
        conn.close()
        return render_template("REGISTER.html", students=students, workshops=workshops, message=message)

In [ ]:
#Task 3.3
@app.route("/REPORT", methods=ac)
def report():
    if method == "POST":
        return redirect({{url_for("REPORT.html")}})
    else:
        #retrieve workshop id, title, max places and student count 
        workshops = getdb().execute("""
        SELECT Workshop.WorkshopID, Workshop.Title, Workshop.Maxplaces, COUNT(*) AS studentcount
        FROM Workshop INNER JOIN Registrations 
        ON Workshop.WorkshopID = Registrations.WorkshopID 
        GROUP BY Registration.StudentID
        ORDER BY studentcount DESC Workshop.Title ASC
        """).fetchall()
        #most popular workshop is title of first row, since studentcount is in descending order
        most_popular_workshop = workshop[0][3]
        return render_template("REPORT.html", workshops=workshops, most_popular_workshop=most_popular_workshop)

In [ ]:
#run the server
if __name__ == "__main__":
    app.run(port=5000)